# 02 — $Q_{net}$ decomposition from GEOS atmosphere diagnostics

Reconstruct the net downward surface heat flux from the GEOS surface collections,

$$Q_{net} = SW_{net} + LW_{net} - LH - SH,$$

with GEOS/MERRA-2 conventions (radiative terms positive **down**; turbulent terms
`EFLUX`, `HFLUX` positive **up**), and check closure against the ocean-side `oceQnet`
on a common 1° grid.

**Prerequisite**: the flux variable names and collections recorded in notebook 00 —
edit `FLUX_SOURCES` below to match. **Caveats**: the atmosphere (c1440 cubed-sphere) and
ocean (LLC2160) grids differ, so both sides are bin-averaged to 1°; under sea ice
`oceQnet` includes ice–ocean exchange, so closure statistics exclude latitudes poleward
of 60°; the 15-minute `flx` collection is time-averaged while the ocean output is
instantaneous — expect residual scatter from the sampling mismatch.

In [ ]:
# Environment check: run on SciServer (Kraken domain, with the Poseidon DYAMOND
# ceph volume attached), or set DYAMOND_ROOT to a local subset.
from dyamond_fluxes import dyamond_root

root = dyamond_root()  # raises with setup instructions if the data is absent
print(f"DYAMOND root: {root}")

In [ ]:
# variable -> (collection, variable name in that collection); from notebook 00.
FLUX_SOURCES = {
    "latent": ("tavg_15mn_2d_flx_Mx", "EFLUX"),
    "sensible": ("tavg_15mn_2d_flx_Mx", "HFLUX"),
    "shortwave": ("geosgcm_surf", "SWGNT"),
    "longwave": ("geosgcm_surf", "LWGNT"),
}
SNAPSHOT = "2020-07-15T12:00"

In [ ]:
import xarray as xr

from dyamond_fluxes import nearest_file, peek_variables

fields = {}
for name, (coll, var) in FLUX_SOURCES.items():
    path = nearest_file(coll, SNAPSHOT)
    with xr.open_dataset(path) as ds_file:
        if var not in ds_file:
            raise KeyError(
                f"{var!r} not in {path.name}; available: {list(ds_file.data_vars)}. "
                "Update FLUX_SOURCES from notebook 00's inventory."
            )
        fields[name] = ds_file[var].squeeze().load()
    print(f"{name:10s} <- {var:8s} from {path.name}")

In [ ]:
from dyamond_fluxes import qnet_from_components

decomp = qnet_from_components(
    swgnt=fields["shortwave"],
    lwgnt=fields["longwave"],
    eflux=fields["latent"],
    hflux=fields["sensible"],
)
decomp

In [ ]:
from dyamond_fluxes import load_geos_coords

coords = load_geos_coords()
print(coords)

# Adjust these two names to the coordinate file's actual variables (notebook 00).
atm_lon = coords["lons"]
atm_lat = coords["lats"]
atm_lon, atm_lat = xr.broadcast(atm_lon.squeeze(), atm_lat.squeeze())

In [ ]:
from dyamond_fluxes import bin_to_latlon, open_ocean_dataset, to_positive_down

DLON = DLAT = 1.0

binned = {
    name: bin_to_latlon(decomp[name], atm_lon, atm_lat, dlon=DLON, dlat=DLAT)
    for name in ["shortwave", "longwave", "latent", "sensible", "qnet"]
}

ds_ocn = open_ocean_dataset(["oceQnet"])
snap_ocn = ds_ocn.sel(time=SNAPSHOT, method="nearest")
qnet_ocn = to_positive_down(snap_ocn["oceQnet"]).where(ds_ocn["Depth"] > 0)
qnet_ocn_binned = bin_to_latlon(
    qnet_ocn.load(), ds_ocn["XC"], ds_ocn["YC"], area=ds_ocn["rA"], dlon=DLON, dlat=DLAT
)
print("ocean snapshot:", snap_ocn.time.values)

In [ ]:
from pathlib import Path

import cmocean
import matplotlib.pyplot as plt

FIGDIR = Path("../figures")
FIGDIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True, sharey=True)
for ax, name in zip(axes.ravel(), ["shortwave", "longwave", "latent", "sensible"]):
    da = binned[name].where(qnet_ocn_binned.notnull())  # ocean points only
    pc = ax.pcolormesh(da.lon, da.lat, da, cmap=cmocean.cm.balance, vmin=-300, vmax=300)
    ax.set_title(f"{name} (positive down)")
fig.colorbar(pc, ax=axes, shrink=0.8, label="W m$^{-2}$")
fig.suptitle(f"GEOS surface heat flux components, {SNAPSHOT}")
fig.savefig(FIGDIR / "qnet_components_1deg.png", dpi=200, bbox_inches="tight")

In [ ]:
import numpy as np

diff = (binned["qnet"] - qnet_ocn_binned).where(abs(qnet_ocn_binned.lat) < 60)

fig, ax = plt.subplots(figsize=(10, 4.5))
pc = ax.pcolormesh(diff.lon, diff.lat, diff, cmap=cmocean.cm.balance, vmin=-100, vmax=100)
fig.colorbar(pc, ax=ax, label="W m$^{-2}$")
ax.set_title("GEOS (SW+LW-LH-SH) minus ocean oceQnet, 1° bins")
fig.savefig(FIGDIR / "qnet_closure_map.png", dpi=200, bbox_inches="tight")

vals = diff.values[np.isfinite(diff.values)]
print(f"closure residual: mean {vals.mean():.2f}, RMS {np.sqrt((vals**2).mean()):.2f} W m-2")

A modest residual is expected even for a perfect decomposition: instantaneous ocean
output vs. 15-minute/hourly time-averaged atmosphere collections, different grids, and
coupler-side adjustments (under-ice fluxes, snow/runoff enthalpy). Large systematic
patterns instead indicate a sign-convention or variable-selection error — recheck the
`long_name` conventions recorded in notebook 00.